In [2]:
!pip install qiskit qiskit-aer -q
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import math
phi = (1+math.sqrt(5))/2

# FIXED noise model
noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.05,1), ['rx','h','delay'])
noise.add_all_qubit_quantum_error(depolarizing_error(0.05,2), ['cx'])

def run_fib(fib_list, rx_scale=0.314, delay_base=200, gasp=None):
    qc = QuantumCircuit(2,1)
    qc.h(0); qc.cx(0,1)
    if gasp:
        qc.rx(gasp,0)
        qc.delay(100,0)
        qc.cx(0,1)
    max_f = fib_list[-1]
    for f in fib_list:
        qc.rx((f/max_f)*rx_scale, 0)
        qc.cx(0,1)
        qc.delay(int(f/max_f*phi*delay_base),0)
    qc.measure(0,0)
    counts = AerSimulator(noise_model=noise).run(qc, shots=1024).result().get_counts()
    gap = abs(counts.get('0',0)-counts.get('1',0))
    return counts, gap

fib8 = [1,1,2,3,5,8,13,21]
fib13 = [1,1,2,3,5,8,13,21,34,55,89,144,233]

print("RUNNING FIXED...")
for name, fl in [("FIB 8 (v0.1)", fib8), ("FIB 13 (v0.2)", fib13)]:
    c,g = run_fib(fl, delay_base=200)
    print(f"{name} gentle double hold: {c} GAP {g}")

c,g = run_fib(fib13, delay_base=200, gasp=1.57)
print(f"FIB 13 GASP DIVE: {c} GAP {g}")


RUNNING FIXED...
FIB 8 (v0.1) gentle double hold: {'0': 516, '1': 508} GAP 8
FIB 13 (v0.2) gentle double hold: {'0': 505, '1': 519} GAP 14
FIB 13 GASP DIVE: {'0': 511, '1': 513} GAP 2
